# Настройка DuckLake

In [1]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = True
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 5
%config SqlMagic.named_parameters = "enabled"

In [2]:
import duckdb
import pandas as pd

%load_ext sql
conn = duckdb.connect()
%sql conn --alias duckdb

/Users/i.korsakov/_code/github/pet_project_what_is_ducklake/.venv/lib/python3.13/site-packages/sql/parse.py:338: SyntaxWarning: invalid escape sequence '\:'
  """
/Users/i.korsakov/_code/github/pet_project_what_is_ducklake/.venv/lib/python3.13/site-packages/sql/parse.py:368: SyntaxWarning: invalid escape sequence '\:'
  """


The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

In [3]:
%sql duckdb:///:memory:

Connecting and switching to connection 'duckdb:///:memory:'

# Создание таблицы в DuckLake

In [4]:
%%sql
INSTALL ducklake;

,Success


In [5]:
%%sql
ATTACH 'ducklake:my_ducklake.ducklake' AS my_ducklake;
USE my_ducklake;

,Success


In [6]:
%%sql
INSTALL fakeit FROM community;
LOAD fakeit;

CREATE OR REPLACE TABLE fake_data AS
SELECT
    s.id AS id,
    fakeit_name_full() AS name,
    fakeit_contact_email() AS email,
    fakeit_address_city() AS city,
    fakeit_address_country() AS country
FROM
    generate_series(1, 100) AS s(id);

,Success


In [7]:
%%sql
FROM fake_data

,id,name,email,city,country
0,1,Wellington DuBuque,damariscummings@wolff.name,Turcottechester,Nigeria
1,2,Troy Reichel,williamgutkowski@huels.info,Corneliusstad,Jordan
2,3,Annette DuBuque,eleanorehermiston@hilpert.io,Millsfort,Tunisia
3,4,Russel Zboncak,alvinapfeffer@mccullough.com,Kattiechester,Chad
4,5,Jerrold Predovic,roxanerohan@brekke.name,Schmittview,Austria
...,...,...,...,...,...
95,96,Adolphus Jerde,zitamohr@hoeger.com,Simland,Suriname
96,97,Nickolas Boyer,courtneyhomenick@nienow.name,Quigleymouth,Croatia
97,98,Ignatius Beer,marceldickinson@herman.biz,Leolaside,Maldives
98,99,Jarod Bauch,lonzoswift@hilll.net,Bernierfurt,Belize


# Спецификация DuckLake

[Specification/Tables](https://ducklake.select/docs/stable/specification/tables/overview)

![](https://ducklake.select/images/schema/ducklake-schema-v1.0-light.svg)

In [8]:
%%sql
SELECT
    table_catalog,
    table_name
FROM information_schema.tables

,table_catalog,table_name
0,__ducklake_metadata_my_ducklake,ducklake_column
1,__ducklake_metadata_my_ducklake,ducklake_column_mapping
2,__ducklake_metadata_my_ducklake,ducklake_column_tag
3,__ducklake_metadata_my_ducklake,ducklake_data_file
4,__ducklake_metadata_my_ducklake,ducklake_delete_file
5,__ducklake_metadata_my_ducklake,ducklake_files_scheduled_for_deletion
6,__ducklake_metadata_my_ducklake,ducklake_file_column_stats
7,__ducklake_metadata_my_ducklake,ducklake_file_partition_value
8,__ducklake_metadata_my_ducklake,ducklake_file_variant_stats
9,__ducklake_metadata_my_ducklake,ducklake_inlined_data_1_1


In [9]:
%%sql
SELECT current_catalog()

,current_catalog()
0,my_ducklake


In [23]:
%%sql
USE '__ducklake_metadata_my_ducklake'

,Success


In [11]:
%%sql
FROM ducklake_snapshot_changes

,snapshot_id,changes_made,author,commit_message,commit_extra_info
0,3,"dropped_table:2,created_table:""main"".""fake_dat...",None,None,None
1,4,altered_table:3,None,None,None
2,5,"inserted_into_table:3,deleted_from_table:3",None,None,None
3,6,"dropped_table:3,created_table:""main"".""fake_dat...",None,None,None
4,7,"dropped_table:4,created_table:""main"".""fake_dat...",None,None,None
5,8,"dropped_table:5,created_table:""main"".""fake_dat...",None,None,None


In [12]:
%%sql
FROM ducklake_snapshot

,snapshot_id,snapshot_time,schema_version,next_catalog_id,next_file_id
0,3,2026-05-27 13:31:39.962865+03:00,3,4,3
1,4,2026-05-27 14:04:07.638247+03:00,4,4,3
2,5,2026-05-27 14:07:03.205849+03:00,4,4,4
3,6,2026-05-27 14:45:15.790108+03:00,5,5,5
4,7,2026-05-27 15:05:24.448127+03:00,6,6,6
5,8,2026-05-27 15:05:41.326321+03:00,7,7,7


In [13]:
%%sql
FROM ducklake_data_file

,data_file_id,table_id,begin_snapshot,end_snapshot,file_order,path,path_is_relative,file_format,record_count,file_size_bytes,footer_size,row_id_start,partition_id,encryption_key,mapping_id,partial_max
0,2,3,3,5,<NA>,ducklake-019e68fd-6ebc-7215-8817-196d95def1c7....,True,parquet,100,7304,665,0,<NA>,None,<NA>,<NA>
1,3,3,5,6,<NA>,ducklake-019e691f-fc3e-7e60-973f-409e0847b18a....,True,parquet,100,8162,905,100,<NA>,None,<NA>,<NA>
2,4,4,6,7,<NA>,ducklake-019e6940-d010-7872-93ad-322d79be8ed0....,True,parquet,100,7269,671,0,<NA>,None,<NA>,<NA>
3,5,5,7,8,<NA>,ducklake-019e6953-4161-7231-aa9f-b5212820fa74....,True,parquet,100,7331,667,0,<NA>,None,<NA>,<NA>
4,6,6,8,<NA>,<NA>,ducklake-019e6953-8350-7fe6-a844-488e5c8b442a....,True,parquet,100,7221,689,0,<NA>,None,<NA>,<NA>


In [19]:
df = %sql FROM ducklake_data_file WHERE end_snapshot IS NULL

In [20]:
pd.read_parquet(f'my_ducklake.ducklake.files/main/fake_data/{df.path[0]}')

,id,name,email,city,country
0,1,Eunice Farrell,marjorycorwin@kohler.info,Faheyberg,Serbia
1,2,Armando Hills,naomietremblay@blanda.info,Elouisechester,Spain
2,3,Morris Simonis,stephanauer@green.com,Mooreberg,Palau
3,4,Haskell VonRueden,haileefadel@baumbach.net,Caitlynshire,China
4,5,Polly Marvin,camylletowne@emard.info,Mrazfort,Georgia
...,...,...,...,...,...
95,96,Therese Mueller,genesisvandervort@blick.biz,Kyrafurt,Germany
96,97,Makayla Gutmann,julestreutel@kuvalis.org,Donnellyhaven,Greece
97,98,Freeman Paucek,willardemard@johns.io,Lewisstad,Angola
98,99,Vladimir Emmerich,alisaeffertz@mayer.info,Schinnerhaven,Honduras


# Уборка в DuckLake
- [Expire Snapshots](https://ducklake.select/docs/stable/duckdb/maintenance/expire_snapshots)
- [Cleanup of Files](https://ducklake.select/docs/stable/duckdb/maintenance/cleanup_of_files)

In [25]:
%%sql
FROM ducklake_snapshot

,snapshot_id,snapshot_time,schema_version,next_catalog_id,next_file_id
0,3,2026-05-27 13:31:39.962865+03:00,3,4,3
1,4,2026-05-27 14:04:07.638247+03:00,4,4,3
2,5,2026-05-27 14:07:03.205849+03:00,4,4,4
3,6,2026-05-27 14:45:15.790108+03:00,5,5,5
4,7,2026-05-27 15:05:24.448127+03:00,6,6,6
5,8,2026-05-27 15:05:41.326321+03:00,7,7,7


In [31]:
%%sql
CALL ducklake_expire_snapshots('my_ducklake', older_than => now() - INTERVAL '10 minute');

,Success


In [32]:
%%sql
FROM ducklake_snapshot

,snapshot_id,snapshot_time,schema_version,next_catalog_id,next_file_id
0,8,2026-05-27 15:05:41.326321+03:00,7,7,7


In [33]:
%%sql
CALL ducklake_cleanup_old_files(
    'my_ducklake',
    cleanup_all => true
);

,Success


In [34]:
%%sql
SELECT current_catalog()

,current_catalog()
0,__ducklake_metadata_my_ducklake


# Изменение схемы (модели)
- [Schema Evolution](https://ducklake.select/docs/stable/duckdb/usage/schema_evolution)

In [35]:
%%sql
USE 'my_ducklake';

,Success


In [36]:
%%sql
ALTER TABLE fake_data
ADD COLUMN name_prefix VARCHAR;

,Success


In [37]:
%%sql
from fake_data

,id,name,email,city,country,name_prefix
0,1,Eunice Farrell,marjorycorwin@kohler.info,Faheyberg,Serbia,None
1,2,Armando Hills,naomietremblay@blanda.info,Elouisechester,Spain,None
2,3,Morris Simonis,stephanauer@green.com,Mooreberg,Palau,None
3,4,Haskell VonRueden,haileefadel@baumbach.net,Caitlynshire,China,None
4,5,Polly Marvin,camylletowne@emard.info,Mrazfort,Georgia,None
...,...,...,...,...,...,...
95,96,Therese Mueller,genesisvandervort@blick.biz,Kyrafurt,Germany,None
96,97,Makayla Gutmann,julestreutel@kuvalis.org,Donnellyhaven,Greece,None
97,98,Freeman Paucek,willardemard@johns.io,Lewisstad,Angola,None
98,99,Vladimir Emmerich,alisaeffertz@mayer.info,Schinnerhaven,Honduras,None


In [38]:
%%sql
UPDATE fake_data
SET name_prefix = fakeit_name_prefix()

,Success


In [39]:
%%sql
from fake_data

,id,name,email,city,country,name_prefix
0,1,Eunice Farrell,marjorycorwin@kohler.info,Faheyberg,Serbia,Mr.
1,2,Armando Hills,naomietremblay@blanda.info,Elouisechester,Spain,Dr.
2,3,Morris Simonis,stephanauer@green.com,Mooreberg,Palau,Dr.
3,4,Haskell VonRueden,haileefadel@baumbach.net,Caitlynshire,China,Miss
4,5,Polly Marvin,camylletowne@emard.info,Mrazfort,Georgia,Miss
...,...,...,...,...,...,...
95,96,Therese Mueller,genesisvandervort@blick.biz,Kyrafurt,Germany,Miss
96,97,Makayla Gutmann,julestreutel@kuvalis.org,Donnellyhaven,Greece,Ms.
97,98,Freeman Paucek,willardemard@johns.io,Lewisstad,Angola,Mr.
98,99,Vladimir Emmerich,alisaeffertz@mayer.info,Schinnerhaven,Honduras,Mrs.


# Time travel

In [40]:
%%sql
SELECT current_catalog()

,current_catalog()
0,my_ducklake


In [41]:
%%sql
USE '__ducklake_metadata_my_ducklake'

,Success


In [42]:
%%sql
FROM ducklake_snapshot_changes

,snapshot_id,changes_made,author,commit_message,commit_extra_info
0,8,"dropped_table:5,created_table:""main"".""fake_dat...",None,None,None
1,9,altered_table:6,None,None,None
2,10,"inserted_into_table:6,deleted_from_table:6",None,None,None


In [43]:
%%sql
FROM ducklake_snapshot

,snapshot_id,snapshot_time,schema_version,next_catalog_id,next_file_id
0,8,2026-05-27 15:05:41.326321+03:00,7,7,7
1,9,2026-05-27 15:18:44.985028+03:00,8,7,7
2,10,2026-05-27 15:18:46.361450+03:00,8,7,8


In [44]:
%%sql
USE 'my_ducklake';

,Success


In [50]:
%%sql
SELECT * FROM fake_data AT (VERSION => 10);

,id,name,email,city,country,name_prefix
0,1,Eunice Farrell,marjorycorwin@kohler.info,Faheyberg,Serbia,Mr.
1,2,Armando Hills,naomietremblay@blanda.info,Elouisechester,Spain,Dr.
2,3,Morris Simonis,stephanauer@green.com,Mooreberg,Palau,Dr.
3,4,Haskell VonRueden,haileefadel@baumbach.net,Caitlynshire,China,Miss
4,5,Polly Marvin,camylletowne@emard.info,Mrazfort,Georgia,Miss
...,...,...,...,...,...,...
95,96,Therese Mueller,genesisvandervort@blick.biz,Kyrafurt,Germany,Miss
96,97,Makayla Gutmann,julestreutel@kuvalis.org,Donnellyhaven,Greece,Ms.
97,98,Freeman Paucek,willardemard@johns.io,Lewisstad,Angola,Mr.
98,99,Vladimir Emmerich,alisaeffertz@mayer.info,Schinnerhaven,Honduras,Mrs.


## 